In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [2]:
from transformers import AutoTokenizer, LlamaForCausalLM

llama_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B", use_fast=True)
llama_model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")
llama_model = llama_model.to(device)

/Users/alibayram/.pyenv/versions/3.13.3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
token = llama_tokenizer.encode("ay")
token_tensor = torch.tensor([[token[1]]]).to(device)
token_tensor

tensor([[352]], device='mps:0')

In [4]:
embedding = llama_model.model.embed_tokens(token_tensor)
embedding.shape

torch.Size([1, 1, 2048])

In [5]:
from turkish_tokenizer import core as tt

tokens, ids = tt.tokenize("Merhaba dünya")
print(tokens)  # ['merhaba', '<space>', 'dünya']
print(ids)     # [2036, 1, 224]

['<uppercase>', 'merhaba', '<space>', 'dünya']
[0, 2036, 1, 224]


In [6]:
vocab_dict = {**tt.bpe_tokens, **tt.suffixes, **tt.roots}

vocab_size = 32768
vocab_size

32768

In [7]:
vocab_dict["<uppercase>"]

0

In [8]:
from llama3_2.config import LlamaConfig
teacher_config = LlamaConfig()


In [9]:
from llama3_2.model_trfmrs import LlamaModel


teacher_model = LlamaModel(teacher_config)
teacher_model = teacher_model.to(device)
teacher_model

LlamaModel(
  (embed_tokens): Embedding(128256, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [10]:
teacher_model.load_state_dict(llama_model.model.state_dict())

<All keys matched successfully>

In [11]:
def count_parameters(model):
    """Count the number of parameters in a model"""
    total_params = sum(p.numel() for p in model.parameters())
    return total_params

In [12]:
# Count parameters for the llama_model
total_params = count_parameters(llama_model.model)
print(f"Total parameters in llama_model: {total_params:,}")

Total parameters in llama_model: 1,235,814,400


In [13]:
total_params = count_parameters(teacher_model)
print(f"Total parameters in teacher_model: {total_params:,}")

Total parameters in teacher_model: 1,235,814,400


In [14]:
llama_model_out = llama_model.model(token_tensor)
llama_model_out

BaseModelOutputWithPast(last_hidden_state=tensor([[[-0.0588,  0.0411,  1.4992,  ...,  0.4604, -0.5548,  0.2668]]],
       device='mps:0', grad_fn=<MulBackward0>), past_key_values=<transformers.cache_utils.DynamicCache object at 0x3125d3a10>, hidden_states=None, attentions=None)

In [15]:
teacher_model_output = teacher_model(token_tensor)
teacher_model_output

tensor([[[-0.0609,  0.0436,  1.5031,  ...,  0.4612, -0.5579,  0.2677]]],
       device='mps:0', grad_fn=<MulBackward0>)

## Map Embeddings

In [16]:
def tr_capitalize(word):
  # i -> İ
  if word[0] == "i":
    return "İ" + word[1:]
  else:
    return word.capitalize()

In [17]:
t1 = [1, 2, 3]
t2 = [4, 5, 6]

t = [t1, t2]
t = torch.tensor(t, dtype=torch.float32)

t = torch.mean(t, dim=0)
t

tensor([2.5000, 3.5000, 4.5000])

In [18]:
import torch.nn.functional as F


def get_vector_for_token(token, model, tokenizer):
  token_id = 0
  if token == "<uppercase>":
    token_id = 128002
  elif token == "<space>":
    token_id = 128003
  elif token == "<newline>":
    token_id = 128011
  elif token == "<tab>":
    token_id = 128012
  elif token == "<unknown>":
    token_id = 128013
  elif token.startswith("kok_") or token.startswith("ek_") or token.startswith("special_"):
    token_id = 128014
  
  if token_id > 0:
    return model.embed_tokens(torch.tensor([token_id]).to(device))

  token_ids = tokenizer.encode(token)
  token_ids0 = tokenizer.encode(tr_capitalize(token))

  if (len(token_ids)) > (len(token_ids0)):
    token_ids0.remove(128000)
    # mean of token_ids0
    token_ids0 = torch.tensor(token_ids0).to(device)
    token_vectors = model.embed_tokens(token_ids0)
    return torch.mean(token_vectors, dim=0)
  else:
    token_ids.remove(128000)
    # mean of token_ids
    token_ids = torch.tensor(token_ids).to(device)
    token_vectors = model.embed_tokens(token_ids)
    return torch.mean(token_vectors, dim=0).to(device)

  

In [19]:
student_config = LlamaConfig(vocab_size=vocab_size)
embeddings = teacher_model.embed_tokens.weight[:vocab_size]
student_model = LlamaModel(student_config, embeddings)
student_model = student_model.to(device)
student_model.layers.load_state_dict(teacher_model.layers.state_dict())
student_model.norm.load_state_dict(teacher_model.norm.state_dict())
student_model

LlamaModel(
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [38]:
v = get_vector_for_token("a", teacher_model, llama_tokenizer)
v.shape

torch.Size([2048])

In [21]:
torch.sum(v)

tensor(0.1318, device='mps:0', grad_fn=<SumBackward0>)

In [22]:
v[3] = v[2]
v

tensor([ 0.0247,  0.0017, -0.0154,  ..., -0.0230, -0.0046,  0.0060],
       device='mps:0', grad_fn=<CopySlices>)

In [23]:
i = 3241
token = list(vocab_dict.keys())[i]
token_id = vocab_dict[token]

token, token_id

('durumu', 25810)

In [24]:
from tqdm import tqdm

for i in tqdm(range(len(vocab_dict.keys())), desc="Mapping embeddings to Llama embeddings"):
  token = list(vocab_dict.keys())[i]
  token_id = vocab_dict[token]
  try:
    v = get_vector_for_token(token, teacher_model, llama_tokenizer)
    student_model.embed_tokens.data[token_id] = v
  except Exception as e:
    print(e)
    print(token, token_id)
    break


Mapping embeddings to Llama embeddings:   0%|          | 0/31356 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Mapping embeddings to Llama embeddings: 100%|██████████| 31356/31356 [00:14<00:00, 2146.63it/s]


In [25]:
teacher_model.eval()
student_model.eval()

LlamaModel(
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm()
      (post_attention_layernorm): LlamaRMSNorm()
    )
  )
  (norm): LlamaRMSNorm()
)

In [36]:
new_token_tensor = llama_tokenizer.encode("a")[1:]
new_token_tensor = torch.tensor([new_token_tensor]).to(device)

teacher_model_output = teacher_model(new_token_tensor)
student_model_out = student_model(new_token_tensor)

print(teacher_model_output.shape, student_model_out.shape, torch.mean(teacher_model_output, dim=1), torch.mean(student_model_out, dim=1))


torch.Size([1, 1, 2048]) torch.Size([1, 1, 2048]) tensor([[ 0.1447, -0.1535,  2.1087,  ...,  0.4828, -0.1266,  0.6782]],
       device='mps:0', grad_fn=<MeanBackward1>) tensor([[ 0.1447, -0.1535,  2.1087,  ...,  0.4828, -0.1266,  0.6782]],
       device='mps:0', grad_fn=<MeanBackward1>)


In [27]:
count_parameters(teacher_model), count_parameters(student_model)

(1235814400, 1040254976)

In [28]:
""" import safetensors


safetensors.torch.save_file(teacher_model.state_dict(), "teacher_model.safetensors")
safetensors.torch.save_file(student_model.state_dict(), "student_model.safetensors") """

' import safetensors\n\n\nsafetensors.torch.save_file(teacher_model.state_dict(), "teacher_model.safetensors")\nsafetensors.torch.save_file(student_model.state_dict(), "student_model.safetensors") '